# Drone Swarm Robotics - v3: Deep Learning, Reinforcement Learning & Robotics
Everything runs in Colab. Pretrained weights are loaded from the repo's `models/` folder, so each section runs in seconds;
every section also has an optional *train from scratch* cell.

| # | Topic | Technique |
|---|---|---|
| 1 | Multi-agent navigation | MAPPO (PPO, centralised critic) with an attention policy + CBF safety shield |
| 2 | Decentralised flocking | Graph Neural Network, behaviour cloning + DAgger |
| 3 | Probabilistic search | Double + Dueling DQN on a Bayesian belief map |
| 4 | Survivor detection | CNN on aerial images, ROC, altitude trade-off |
| 5 | Intruder prediction | LSTM & Transformer sequence models, prediction-aware avoidance |
| 6 | Fault detection | Autoencoder anomaly detection on telemetry |
| 7 | Model-based control | Neural dynamics model + MPPI |
| 8 | Classical robotics | Kalman filter, particle filter, A*, RRT*, control barrier functions, occupancy mapping |

In [ ]:
!git clone -q https://github.com/guptuv/drone-swarm-robotics.git 2>/dev/null || (cd drone-swarm-robotics && git pull -q)
%cd drone-swarm-robotics
!pip -q install -e .
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from IPython.display import HTML, display
from matplotlib import animation, rc
rc('animation', html='jshtml')
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())

## 1. Multi-agent RL: MAPPO + CBF safety shield
Eight drones on a circle swap to the opposite side through obstacles. Each drone runs the *same* small attention network on local observations (k nearest drones + obstacles).

In [ ]:
from drone_swarm.ai import marl
pol = marl.AttentionPolicy(5, 3); pol.load_state_dict(torch.load('models/mappo_swarm_nav.pt'))
rows = []
for name, c in [('potential field', marl.PotentialFieldController()), ('MAPPO', pol), ('MAPPO + CBF shield', marl.CBFShield(pol))]:
    r = marl.evaluate(c, n_agents=8, episodes=64)
    rows.append(dict(controller=name, success=r['success_rate'], collision_free=r['collision_free_eps'], time_s=r['mean_time_s']))
pd.DataFrame(rows).round(3)

In [ ]:
r = marl.evaluate(marl.CBFShield(pol), n_agents=10, episodes=1, seed=3, record=True); T = r['traj']
fig, a = plt.subplots(figsize=(5, 5)); a.set_xlim(0, 40); a.set_ylim(0, 40); a.set_aspect('equal')
for k, (x, y) in enumerate(r['obs_c']):
    if r['obs_r'][k] > 0: a.add_patch(plt.Circle((x, y), r['obs_r'][k], color='gray', alpha=.5))
d = a.scatter(*T[0].T, c=range(T.shape[1]), cmap='tab10', s=50); a.scatter(*r['goal'].T, marker='*', c='gold', s=80)
anim = animation.FuncAnimation(fig, lambda f: d.set_offsets(T[f]), frames=len(T), interval=80); plt.close(); anim

In [ ]:
# optional: train from scratch (about 15-20 min on Colab CPU)
# pol, hist = marl.train_mappo(updates=400, log_every=20, env_kw=dict(w_hit_obs=10.0), eval_every=20)

## 2. Graph Neural Network flocking (imitation learning)

In [ ]:
from drone_swarm.ai import gnn
g = gnn.FlockGNN(); g.load_state_dict(torch.load('models/gnn_flocking.pt'))
pd.DataFrame(gnn.compare(g, sizes=(20, 40, 80), episodes=5)).pivot(index='n', columns='controller', values='cost_mean').round(3)

In [ ]:
# optional: g, hist = gnn.train_gnn(epochs=30)

## 3. DQN for probabilistic search

In [ ]:
from drone_swarm.ai import dqn_search
q = dqn_search.QNet(); q.load_state_dict(torch.load('models/dqn_search.pt'))
res = {p: dqn_search.run_policy(p, episodes=256, model=q) for p in ['random', 'lawnmower', 'greedy', 'dqn']}
for p, r in res.items(): plt.plot(r['curve'], label=f"{p} ({r['p_detect']:.2f})")
plt.legend(); plt.xlabel('step'); plt.ylabel('P(survivor detected)'); plt.show()

In [ ]:
# optional: q, hist = dqn_search.train_dqn(steps=3000)

## 4. CNN survivor detection

In [ ]:
from drone_swarm.ai import vision
det = vision.DetectorCNN(); det.load_state_dict(torch.load('models/detector_cnn.pt'))
X, y, alt = vision.make_dataset(16, np.random.default_rng(1))
with torch.no_grad(): p = torch.sigmoid(det(torch.tensor(X))).numpy()
fig, ax = plt.subplots(2, 8, figsize=(16, 4.5))
for k, a in enumerate(ax.flat): a.imshow(X[k].transpose(1, 2, 0)); a.axis('off'); a.set_title(f"{'person' if y[k] else 'empty'} {alt[k]:.0f}m p={p[k]:.2f}", fontsize=8)
plt.show()
pd.DataFrame(vision.performance_vs_altitude(det, n=300)).round(3)

In [ ]:
# optional: det, hist = vision.train_detector(n_train=16000, epochs=10)   # use a GPU runtime to speed this up

## 5. LSTM / Transformer trajectory prediction

In [ ]:
from drone_swarm.ai import prediction as pr
lstm = pr.LSTMPredictor(); lstm.load_state_dict(torch.load('models/lstm_predictor.pt'))
tfm = pr.TransformerPredictor(); tfm.load_state_dict(torch.load('models/transformer_predictor.pt'))
Xt, Yt = pr.make_windows(pr.simulate_intruders(60, 300, np.random.default_rng(1)))
with torch.no_grad(): pl, pt = lstm(torch.tensor(Xt)).numpy(), tfm(torch.tensor(Xt)).numpy()
pd.DataFrame({k: pr.ade_fde(v, Yt) for k, v in {'CV': pr.predict_cv(Xt), 'CTRV': pr.predict_ctrv(Xt), 'LSTM': pl, 'Transformer': pt}.items()}, index=['ADE m', 'FDE m']).round(3)

## 6. Autoencoder fault detection

In [ ]:
from drone_swarm.ai import anomaly
ck = torch.load('models/fault_autoencoder.pt', weights_only=False)
fd = anomaly.FaultDetector(); fd.mu, fd.sd, fd.thr = ck['mu'], ck['sd'], ck['thr']
fd.model = anomaly.AutoEncoder(len(fd.mu)); fd.model.load_state_dict(ck['state'])
pd.DataFrame(anomaly.detection_report(fd, runs=2)).round(3)

## 7. Learned dynamics + MPPI

In [ ]:
from drone_swarm.ai import mbrl
dm = mbrl.DynamicsMLP(); dm.load_state_dict(torch.load('models/dynamics_mlp.pt'))
for name, ctrl in [('PD', mbrl.pd_controller), ('MPPI learned', mbrl.MPPI(dm).act)]:
    r = mbrl.fly_slalom(ctrl, seed=0); plt.plot(r['path'][:, 0], r['path'][:, 1], label=f"{name} (clearance {r['min_clearance']:.2f} m)")
for x, y_, rr in mbrl.PILLARS: plt.gca().add_patch(plt.Circle((x, y_), rr, color='gray'))
plt.gca().set_aspect('equal'); plt.legend(); plt.show()

## 8. Classical robotics

In [ ]:
from drone_swarm.robotics import estimation, planning, cbf, mapping
from drone_swarm import make_city
kf = estimation.run_kf(estimation.simulate_flight()); print(f"Kalman RMSE {kf['rmse_kf']:.2f} m vs GPS {kf['rmse_gps']:.2f} m | bias estimate {kf['bias_est'].round(3)}")
occ, _ = planning.occupancy_from_buildings(make_city(seed=3, n=7)); pa, _ = planning.astar(occ, (12, 12), (80, 75)); pr_, _, _ = planning.rrt_star(occ, (12, 12), (80, 75))
print(f"A* {planning.path_length(pa):.1f} m | RRT* {planning.path_length(pr_):.1f} m")
s = cbf.antipodal_swap(use_cbf=True); print(f"CBF swap: min distance {s['min_dist'].min():.2f} m, arrived {s['arrived']:.0%}")
ex = mapping.explore(mapping.make_world(), strategy='frontier', steps=200); print(f"Frontier exploration mapped {ex['known'][-1]:.0%} of free space")